In [1]:
print("hello world")

hello world



# Full project
Create a real-time pipeline that 
ingests hourly air quality measurements (PM2.5, PM10, O₃, CO, NO₂, SO₂, and UV index) from the Open-Meteo Air Quality API for Nairobi (−1.286389, 36.817223) and Mombasa (−4.043477, 39.668206), 
stores the data in MongoDB, 
streams the changes through Kafka, 
and loads them into Cassandra for analytics and dashboards.


## Fetch the data and store in mongodb

In [2]:
!pip install schedule

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import requests
import pymongo
import os
from dotenv import load_dotenv
import logging
from datetime import datetime
import schedule
import time


logging.basicConfig(
    level= logging.INFO,
    format = '%(asctime)s | %(levelname)s | %(name)s| %(message)s',
    handlers=[
        logging.FileHandler(f"fetch_store_logs_{datetime.now().strftime('%Y%m%d')}.log"),
        logging.StreamHandler()
    ]
)

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")

PARAMS = {
    'hourly': 'pm2_5,pm10,ozone,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,uv_index',
    "cities": {
        "nairobi": {"lat": -1.286389, "lon": 36.817223},
        "mombasa": {"lat": -4.043477, "lon": 39.668206},
    }
}

BASE_URL = "https://air-quality-api.open-meteo.com/v1/air-quality" 

client = pymongo.MongoClient(MONGO_URI)
db = client.city_air_quality
collection = db.air_quality_stats



def fetch_and_store():
    # replace the minutes and seconds to only get the rime now 
    current_hour = datetime.now().replace(minute=0, second=0, microsecond=0)


    logging.info(f'Getting the weather data at {current_hour}')

    for city, coords in PARAMS["cities"].items():
        logging.info(f'Getting data for {city}')
        
        params = {}
        params["hourly"] = PARAMS["hourly"]
        params["latitude"] = coords["lat"]
        params["longitude"] = coords["lon"]
        response = requests.get(BASE_URL, params=params )

        if response.status_code != 200:
            logging.debug("failed with the url")
            return
        
        data = response.json()
        logging.info(f"Retreived data {data}")
    
        logging.info("Simulating an hour result of data")
        
        for i, timestamp in enumerate(data["hourly"]["time"]):

            record_time = datetime.fromisoformat(timestamp).replace()

            
            record = {
                    "city": city,
                    "timestamp": record_time,
                    "pm2_5": data["hourly"]["pm2_5"][i],
                    "pm10": data["hourly"]["pm10"][i],
                    "ozone": data["hourly"]["ozone"][i],
                    "carbon_monoxide": data["hourly"]["carbon_monoxide"][i],
                    "nitrogen_dioxide": data["hourly"]["nitrogen_dioxide"][i],
                    "sulphur_dioxide": data["hourly"]["sulphur_dioxide"][i],
                    "uv_index": data["hourly"]["uv_index"][i],
                }

                # insert only when the record doesnt exist 
            if not collection.find_one({"city": city,"timestamp": record_time}):
                collection.insert_one(record)
                logging.info(f"inserted record {record}")
            else:
                logging.info(f'the record already exists try again in the next hour ')


# --- Schedule every hour ---
schedule.every(1).hours.do(fetch_and_store)

# Run immediately at start
fetch_and_store()

# Keep running
while True:
    schedule.run_pending()
    time.sleep(600)

2025-10-19 12:55:14,508|INFO|root|Getting the weather data at 2025-10-19 12:00:00
2025-10-19 12:55:14,509|INFO|root|Getting data for nairobi
2025-10-19 12:55:15,219|INFO|root|Retreived data {'latitude': -1.2999954, 'longitude': 36.800003, 'generationtime_ms': 0.28383731842041016, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 1671.0, 'hourly_units': {'time': 'iso8601', 'pm2_5': 'μg/m³', 'pm10': 'μg/m³', 'ozone': 'μg/m³', 'carbon_monoxide': 'μg/m³', 'nitrogen_dioxide': 'μg/m³', 'sulphur_dioxide': 'μg/m³', 'uv_index': ''}, 'hourly': {'time': ['2025-10-19T00:00', '2025-10-19T01:00', '2025-10-19T02:00', '2025-10-19T03:00', '2025-10-19T04:00', '2025-10-19T05:00', '2025-10-19T06:00', '2025-10-19T07:00', '2025-10-19T08:00', '2025-10-19T09:00', '2025-10-19T10:00', '2025-10-19T11:00', '2025-10-19T12:00', '2025-10-19T13:00', '2025-10-19T14:00', '2025-10-19T15:00', '2025-10-19T16:00', '2025-10-19T17:00', '2025-10-19T18:00', '2025-10-19T19:00', '2025-10-19

KeyboardInterrupt: 

In [3]:
!pip install confluent-kafka cassandra-driver

Defaulting to user installation because normal site-packages is not writeable
  Using cached confluent_kafka-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached cassandra_driver-3.29.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.2 kB)
  Using cached tomli-2.3.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached geomet-0.2.1.post1-py3-none-any.whl.metadata (1.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 611.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 480.3 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.1/250.1 kB 553.7 kB/s eta 0:00:00a 0:00:01


In [ ]:
# this scrip consumes messages from kafka and stores them in Astra DB, however it doesnt define the table structure before hand which lead to unreadale data in the db(It was harder to query the data later)


from confluent_kafka import Consumer
from astrapy import DataAPIClient
import json
import time
import os
from dotenv import load_dotenv
import logging
from datetime import datetime

logging.basicConfig(
    level= logging.INFO,
    format = '%(asctime)s | %(levelname)s | %(name)s| %(message)s',
    handlers=[
        logging.FileHandler(f"consume_to_cassandra_{datetime.now().strftime('%Y%m%d')}.log"),
        logging.StreamHandler()
    ]
)

load_dotenv()
#  Astra DB setup 
ASTRA_DB_TOKEN = os.getenv("ASTRA_DB_TOKEN")
ASTRA_DB_ENDPOINT = os.getenv("ASTRA_DB_ENDPOINT")
ASTRA_COLLECTION = "air_quality"

# kafka setup
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
CONFLUENT_API_KEY = os.getenv("CONFLUENT_API_KEY")
CONFLUENT_API_SECRET = os.getenv("CONFLUENT_API_SECRET")


client = DataAPIClient(ASTRA_DB_TOKEN)
db = client.get_database_by_api_endpoint(ASTRA_DB_ENDPOINT)

# create the collection if it doesn't exist
if ASTRA_COLLECTION not in db.list_collection_names():
    db.create_collection(ASTRA_COLLECTION)
collection = db.get_collection(ASTRA_COLLECTION)

logging.info(f" Connected to Astra DB: {ASTRA_COLLECTION}")
print(os.getenv("KAFKA_BOOTSTRAP_SERVERS"))
# Kafka consumer setup 
conf = {
    'bootstrap.servers': KAFKA_BOOTSTRAP_SERVERS,  # Replace with your Kafka bootstrap server
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': CONFLUENT_API_KEY,
    'sasl.password': CONFLUENT_API_SECRET,
    "group.id": "airquality-consumer-group", 
    "auto.offset.reset": "earliest",

}

consumer = Consumer(conf)
topic = "air_quality.city_air_quality.air_quality_stats.city_air_quality.air_quality_stats"
consumer.subscribe([topic])

logging.info(f" Listening for messages on '{topic}'...")

#  Consume and insert 
while True:
    msg = consumer.poll(1.0)
    if msg is None:
        continue
    if msg.error():
        logging.info("Kafka error:", msg.error())
        continue

    try:
        logging.info("Message received: loading into astra")
        data = json.loads(msg.value().decode("utf-8"))

        print(data)
        print(type(data))

        # ensure timestamps are stored as strings
        record = {
            "city": data["fullDocument"]["city"],
            "timestamp": data["fullDocument"]["timestamp"],
            "pm2_5": data["fullDocument"]["pm2_5"],
            "pm10": data["fullDocument"]["pm10"],
            "ozone": data["fullDocument"]["ozone"],
            "carbon_monoxide": data["fullDocument"]["carbon_monoxide"],
            "nitrogen_dioxide": data["fullDocument"]["nitrogen_dioxide"],
            "sulphur_dioxide": data["fullDocument"]["sulphur_dioxide"],
            "uv_index": data["fullDocument"]["uv_index"],
        }

        # insert into Astra DB
        collection.insert_one(record)
        logging.info(f" Inserted record for {record['city']} at {record['timestamp']}")

    except Exception as e:
        logging.info("Error inserting record:", e)

    time.sleep(0.5)



2025-10-16 20:21:30,789 | INFO | astrapy.data.database| findCollections
2025-10-16 20:21:31,705 | INFO | httpx| HTTP Request: POST https://5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.apps.astra.datastax.com/api/json/v1/default_keyspace "HTTP/1.1 200 OK"
2025-10-16 20:21:31,707 | INFO | astrapy.data.database| finished findCollections
2025-10-16 20:21:31,782 | INFO | root|  Connected to Astra DB: air_quality
2025-10-16 20:21:31,841 | INFO | root|  Listening for messages on 'air_quality.city_air_quality.air_quality_stats.city_air_quality.air_quality_stats'...


pkc-921jm.us-east-2.aws.confluent.cloud:9092


%6|1760635296.068|GETSUBSCRIPTIONS|rdkafka#consumer-1| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to bhE2iBxiQlqaVG3haZn3eQ
2025-10-16 20:21:37,329 | INFO | root| Message received: loading into astra
2025-10-16 20:21:37,330 | INFO | astrapy.data.collection| insertOne on 'air_quality'


{'_id': {'_data': '8268F1271E000000012B042C0100296E5A1004B752B856C9FD4A58BF2048F76B7CFB3D463C6F7065726174696F6E54797065003C696E736572740046646F63756D656E744B65790046645F6964006468F1271E2F4D225381F74B17000004'}, 'clusterTime': 1760634654000, 'documentKey': {'_id': '{"$oid": "68f1271e2f4d225381f74b17"}'}, 'fullDocument': {'_id': '{"$oid": "68f1271e2f4d225381f74b17"}', 'carbon_monoxide': 404.0, 'city': 'nairobi', 'nitrogen_dioxide': 14.6, 'ozone': 45.0, 'pm10': 15.2, 'pm2_5': 14.6, 'sulphur_dioxide': 4.7, 'timestamp': 1760644800000, 'uv_index': 0.0}, 'ns': {'coll': 'air_quality_stats', 'db': 'city_air_quality'}, 'operationType': 'insert', 'wallTime': 1760634654651}
<class 'dict'>


2025-10-16 20:21:38,187 | INFO | httpx| HTTP Request: POST https://5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.apps.astra.datastax.com/api/json/v1/default_keyspace/air_quality "HTTP/1.1 200 OK"
2025-10-16 20:21:38,189 | INFO | astrapy.data.collection| finished insertOne on 'air_quality'
2025-10-16 20:21:38,189 | INFO | root|  Inserted record for nairobi at 1760644800000
2025-10-16 20:21:38,690 | INFO | root| Message received: loading into astra
2025-10-16 20:21:38,692 | INFO | astrapy.data.collection| insertOne on 'air_quality'


{'_id': {'_data': '8268F1271F000000012B042C0100296E5A1004B752B856C9FD4A58BF2048F76B7CFB3D463C6F7065726174696F6E54797065003C696E736572740046646F63756D656E744B65790046645F6964006468F1271F2F4D225381F74B18000004'}, 'clusterTime': 1760634655000, 'documentKey': {'_id': '{"$oid": "68f1271f2f4d225381f74b18"}'}, 'fullDocument': {'_id': '{"$oid": "68f1271f2f4d225381f74b18"}', 'carbon_monoxide': 108.0, 'city': 'mombasa', 'nitrogen_dioxide': 2.9, 'ozone': 48.0, 'pm10': 13.9, 'pm2_5': 9.7, 'sulphur_dioxide': 2.5, 'timestamp': 1760644800000, 'uv_index': 0.0}, 'ns': {'coll': 'air_quality_stats', 'db': 'city_air_quality'}, 'operationType': 'insert', 'wallTime': 1760634655745}
<class 'dict'>


2025-10-16 20:21:39,005 | INFO | httpx| HTTP Request: POST https://5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.apps.astra.datastax.com/api/json/v1/default_keyspace/air_quality "HTTP/1.1 200 OK"
2025-10-16 20:21:39,007 | INFO | astrapy.data.collection| finished insertOne on 'air_quality'
2025-10-16 20:21:39,008 | INFO | root|  Inserted record for mombasa at 1760644800000
%6|1760639384.397|FAIL|rdkafka#consumer-1| [thrd:GroupCoordinator]: GroupCoordinator: b9-pkc-921jm.us-east-2.aws.confluent.cloud:9092: Disconnected: connection reset by peer (after 3115379ms in state UP)
%4|1760639384.417|FAIL|rdkafka#consumer-1| [thrd:sasl_ssl://b2-pkc-921jm.us-east-2.aws.confluent.cloud:9092/2]: sasl_ssl://b2-pkc-921jm.us-east-2.aws.confluent.cloud:9092/2: Disconnected: connection reset by peer (after 3115065ms in state UP)
%4|1760639384.667|FAIL|rdkafka#consumer-1| [thrd:sasl_ssl://b5-pkc-921jm.us-east-2.aws.confluent.cloud:9092/5]: sasl_ssl://b5-pkc-921jm.us-east-2.aws.confluent.cloud:9092/5: Disco

In [ ]:
# youll need to set up your cassandra in Astra db and the kafka connector explained in confluence_setup folder to run this script
# downoad the secure connect bundle from astra db and set the path in the .env file
# you can find instructions here 
# https://docs.datastax.com/en/astra-db-classic/databases/secure-connect-bundle.html#:~:text=SCB%20never%20expires.-,Download%20SCBs%20with%20the%20Astra%20Portal,an%20archive%20(zip%20file).

from confluent_kafka import Consumer
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import json
import time
import os
from dotenv import load_dotenv
import logging
from datetime import datetime


logging.basicConfig(
    level=logging.INFO,
    format= '%(asctime)s|%(levelname)s|%(name)s|%(message)s',
    handlers = [
        logging.FileHandler(f'consume_to_cassandra_{datetime.now().strftime('%Y%m%d')}.log'),
        logging.StreamHandler()
    ]
)

load_dotenv()

ASTRA_DB_TOKEN = os.getenv("ASTRA_DB_TOKEN")
# 
ASTRA_DB_BUNDLE = os.getenv("ASTRA_DB_BUNDLE")
KEY_SPACE = os.getenv("KEY_SPACE")

#kafka set up
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
CONFLUENT_API_KEY = os.getenv("CONFLUENT_API_KEY")
CONFLUENT_API_SECRET = os.getenv("CONFLUENT_API_SECRET")

logging.info("Connecting to Astra db...\n")



def consume_and_load():
    cloud_config = {'secure_connect_bundle': ASTRA_DB_BUNDLE}
    auth_provider = PlainTextAuthProvider('token', ASTRA_DB_TOKEN)
    cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
    session = cluster.connect(KEY_SPACE)

    logging.info("Successfully connected")

    # Create table if it doesn't exist
    create_table_query = """
    CREATE TABLE IF NOT EXISTS air_quality_data (
        city text,
        timestamp timestamp,
        pm2_5 double,
        pm10 double,
        ozone double,
        carbon_monoxide double,
        nitrogen_dioxide double,
        sulphur_dioxide double,
        uv_index double,
        PRIMARY KEY (city, timestamp)
    ) WITH CLUSTERING ORDER BY (timestamp DESC)
    """
    session.execute(create_table_query)
    logging.info("Table air_quality_data ready")


    # define the insert query to reuse later
    insert_query = """
    INSERT INTO air_quality_data(
    city, timestamp, pm2_5, pm10, ozone, carbon_monoxide, nitrogen_dioxide, sulphur_dioxide, uv_index
    ) VALUES (?,?,?,?,?,?,?,?,?)
    """

    prepared_statement = session.prepare(insert_query)

    conf = {
        "bootstrap.servers" : KAFKA_BOOTSTRAP_SERVERS,
        "security.protocol": "SASL_SSL",
        "sasl.mechanism": "PLAIN",
        'sasl.username': CONFLUENT_API_KEY,
        'sasl.password': CONFLUENT_API_SECRET,
        "group.id": "airquality-consumer-group",
        "auto.offset.reset": "earliest",
        }

    consumer  = Consumer(conf)

    topic = "air_quality.city_air_quality.air_quality_data"
    consumer.subscribe([topic])

    logging.info(f"Listening to messages at {topic}")


    while True:
        msg = consumer.poll(1.0)
        logging.info(f"message received as ")
        if msg is None:
            continue
        if msg.error():
            logging.info(f"Kafka error: {msg.error()}")
            continue
        try:
            logging.info("Message received: loading into cassandra")
            print(msg.value())
            # decode the json value
            data = json.loads(msg.value().decode("utf-8"))
            data = data["fullDocument"]
            #city, timestamp, pm2_5, pm10, ozone, carbon_monoxide, nitrogen_dioxide, sulphur_dioxide, uv_index
            values = (
                data["city"],
                data["timestamp"],
                data["pm2_5"],
                data["pm10"],
                data["ozone"],
                data["carbon_monoxide"],
                data["nitrogen_dioxide"],
                data["sulphur_dioxide"],
                data["uv_index"],
            )

            session.execute(prepared_statement, values)

            logging.info(f'inserted data for city {data["city"]} at {data["timestamp"]}')

            

        except Exception as e:
            logging.info(f"there was an error logging into cassandra : {e}")
            logging.info(f"Failed values: {values}")
        time.sleep(1)



2025-10-19 17:50:39,635|INFO|root|Connecting to Astra db...

2025-10-19 17:50:41,855|WARNING|cassandra.cluster|Downgrading core protocol version from 66 to 65 for 5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.db.astra.datastax.com:29042:98a2cd7e-fa1b-3950-942e-42af495c4248. To avoid this, it is best practice to explicitly set Cluster(protocol_version) to the version supported by your cluster. http://datastax.github.io/python-driver/api/cassandra/cluster.html#cassandra.cluster.Cluster.protocol_version
2025-10-19 17:50:44,697|WARNING|cassandra.cluster|Downgrading core protocol version from 65 to 5 for 5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.db.astra.datastax.com:29042:98a2cd7e-fa1b-3950-942e-42af495c4248. To avoid this, it is best practice to explicitly set Cluster(protocol_version) to the version supported by your cluster. http://datastax.github.io/python-driver/api/cassandra/cluster.html#cassandra.cluster.Cluster.protocol_version
2025-10-19 17:50:46,516|WARNING|cassandra.cluster|Do

b'{"_id":{"_data":"8268F4FAB2000000012B042C0100296E5A1004B8797457114C4011B0FF2C29AE14F71F463C6F7065726174696F6E54797065003C696E736572740046646F63756D656E744B65790046645F6964006468F4FAB2BAE554530670B139000004"},"clusterTime":1760885426000,"documentKey":{"_id":"{\\"$oid\\": \\"68f4fab2bae554530670b139\\"}"},"fullDocument":{"_id":"{\\"$oid\\": \\"68f4fab2bae554530670b139\\"}","carbon_monoxide":204.0,"city":"nairobi","nitrogen_dioxide":15.9,"ozone":54.0,"pm10":14.1,"pm2_5":13.4,"sulphur_dioxide":4.3,"timestamp":1760893200000,"uv_index":0.0},"ns":{"coll":"air_quality_data","db":"city_air_quality"},"operationType":"insert","wallTime":1760885426084}'


2025-10-19 17:52:07,893|INFO|root|inserted data for city nairobi at 1760893200000
2025-10-19 17:52:08,895|INFO|root|message received as 
2025-10-19 17:52:08,897|INFO|root|Message received: loading into cassandra


b'{"_id":{"_data":"8268F4FAB3000000012B042C0100296E5A1004B8797457114C4011B0FF2C29AE14F71F463C6F7065726174696F6E54797065003C696E736572740046646F63756D656E744B65790046645F6964006468F4FAB3BAE554530670B13A000004"},"clusterTime":1760885427000,"documentKey":{"_id":"{\\"$oid\\": \\"68f4fab3bae554530670b13a\\"}"},"fullDocument":{"_id":"{\\"$oid\\": \\"68f4fab3bae554530670b13a\\"}","carbon_monoxide":135.0,"city":"mombasa","nitrogen_dioxide":2.8,"ozone":55.0,"pm10":12.4,"pm2_5":9.0,"sulphur_dioxide":2.4,"timestamp":1760893200000,"uv_index":0.0},"ns":{"coll":"air_quality_data","db":"city_air_quality"},"operationType":"insert","wallTime":1760885427432}'


2025-10-19 17:52:09,684|INFO|root|inserted data for city mombasa at 1760893200000
2025-10-19 17:52:11,685|INFO|root|message received as 
2025-10-19 17:52:12,688|INFO|root|message received as 
2025-10-19 17:52:13,689|INFO|root|message received as 
2025-10-19 17:52:14,690|INFO|root|message received as 
2025-10-19 17:52:15,691|INFO|root|message received as 
2025-10-19 17:52:16,692|INFO|root|message received as 
2025-10-19 17:52:17,693|INFO|root|message received as 
2025-10-19 17:52:18,695|INFO|root|message received as 
2025-10-19 17:52:19,697|INFO|root|message received as 
2025-10-19 17:52:20,699|INFO|root|message received as 
2025-10-19 17:52:21,701|INFO|root|message received as 
2025-10-19 17:52:22,702|INFO|root|message received as 
2025-10-19 17:52:23,704|INFO|root|message received as 
2025-10-19 17:52:24,706|INFO|root|message received as 
2025-10-19 17:52:25,718|INFO|root|message received as 
2025-10-19 17:52:26,724|INFO|root|message received as 
2025-10-19 17:52:27,726|INFO|root|mess

...

2025-10-19 17:59:56,200|INFO|root|inserted data for city mombasa at 1761256800000
2025-10-19 17:59:57,203|INFO|root|message received as 
2025-10-19 17:59:57,205|INFO|root|Message received: loading into cassandra


b'{"_id":{"_data":"8268F4FBED000000042B042C0100296E5A1004B8797457114C4011B0FF2C29AE14F71F463C6F7065726174696F6E54797065003C696E736572740046646F63756D656E744B65790046645F6964006468F4FBED9A05004C5E0C6A6C000004"},"clusterTime":1760885741000,"documentKey":{"_id":"{\\"$oid\\": \\"68f4fbed9a05004c5e0c6a6c\\"}"},"fullDocument":{"_id":"{\\"$oid\\": \\"68f4fbed9a05004c5e0c6a6c\\"}","carbon_monoxide":119.0,"city":"mombasa","nitrogen_dioxide":2.2,"ozone":51.0,"pm10":10.4,"pm2_5":7.6,"sulphur_dioxide":1.9,"timestamp":1761260400000,"uv_index":0.0},"ns":{"coll":"air_quality_data","db":"city_air_quality"},"operationType":"insert","wallTime":1760885741973}'


2025-10-19 17:59:57,455|INFO|root|inserted data for city mombasa at 1761260400000
2025-10-19 17:59:59,457|INFO|root|message received as 
2025-10-19 18:00:00,458|INFO|root|message received as 
2025-10-19 18:00:01,461|INFO|root|message received as 
2025-10-19 18:00:02,462|INFO|root|message received as 
2025-10-19 18:00:03,464|INFO|root|message received as 
2025-10-19 18:00:04,466|INFO|root|message received as 
2025-10-19 18:00:05,467|INFO|root|message received as 
2025-10-19 18:00:06,469|INFO|root|message received as 
2025-10-19 18:00:07,471|INFO|root|message received as 
2025-10-19 18:00:08,472|INFO|root|message received as 
2025-10-19 18:00:09,474|INFO|root|message received as 
2025-10-19 18:00:10,475|INFO|root|message received as 
2025-10-19 18:00:11,477|INFO|root|message received as 
2025-10-19 18:00:12,479|INFO|root|message received as 
2025-10-19 18:00:13,480|INFO|root|message received as 
2025-10-19 18:00:14,482|INFO|root|message received as 
2025-10-19 18:00:15,484|INFO|root|mess

KeyboardInterrupt: 

%4|1760886503.819|MAXPOLL|rdkafka#consumer-6| [thrd:main]: Application maximum poll interval (300000ms) exceeded by 13ms (adjust max.poll.interval.ms for long-running message processing): leaving group
2025-10-19 19:36:17,218|WARNING|cassandra.connection|Heartbeat failed for connection (132732208059680) to 5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.db.astra.datastax.com:29042:00958fd0-af98-391f-9f51-7e3fe763da5c
2025-10-19 19:36:17,982|WARNING|cassandra.connection|Heartbeat failed for connection (132732231184896) to 5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.db.astra.datastax.com:29042:98a2cd7e-fa1b-3950-942e-42af495c4248
2025-10-19 19:36:20,394|WARNING|cassandra.connection|Heartbeat failed for connection (132732206965936) to 5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3.db.astra.datastax.com:29042:678e6a1d-e5b1-3cba-b949-7bfaf823957f
2025-10-19 19:36:23,066|WARNING|cassandra.connection|Heartbeat failed for connection (132732877953008) to 5197129b-26d1-4f74-a3ed-ce868b1d6fc5-westus3